# Learn a repeatable move from its previous tracking error

A stage repeats the same trajectory many times. Predict what happens if the learning update becomes too aggressive. The horizontal coordinate within each motion is distinct from the trial number. Edit the first cell and rerun.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

memory = .75            # sampled plant pole, 0 < memory < 1
relative_step = .9      # fraction of 2 / largest squared singular value
trials = 30
N = 48                 # commands per move
assert 0 < memory < 1 and 0 < relative_step <= 1.12 and 1 <= trials <= 60 and 12 <= N <= 96

## Lift the causal recurrence onto one trial
Model the stage by $y_{k+1}=a y_k+(1-a)u_k$ with zero initial state. Record $y_1$ to $y_N$ for inputs $u_0$ to $u_{N-1}$. A deterministic output disturbance returns in the same phase of every trial.

In [ ]:
lag = np.arange(N)[:,None]-np.arange(N)[None,:]
G = np.tril((1-memory)*memory**np.maximum(lag,0))
t = np.arange(1,N+1)/N
target = .9*np.sin(np.pi*t)**2 + .13*np.sin(6*np.pi*t)
disturbance = .09*np.sin(2*np.pi*t) + .025*np.cos(8*np.pi*t)
sigma = np.linalg.svd(G,compute_uv=False)
gamma = relative_step*2/sigma[0]**2
print(f"Singular values range {sigma[-1]:.4f} to {sigma[0]:.4f}.")
print(f"Learning step {gamma:.3f}; upper bound for monotone contraction {2/sigma[0]**2:.3f}.")
u_test = np.linspace(-1,1,N); state=0.; sequence=[]
for u in u_test:
    state=memory*state+(1-memory)*u
    sequence.append(state)
assert np.allclose(G@u_test,sequence)

## Apply the adjoint of the forward operator
Corrections use the entire previous record. Compare against the theoretically slowest contraction factor; the reference starts at the measured first error. For an intentionally excessive step study only this linear model.

In [ ]:
u = np.zeros(N)
energies, paths = [], []
for j in range(trials+1):
    y = G@u + disturbance
    error = target-y
    paths.append(y.copy())
    energies.append(np.linalg.norm(error)/np.sqrt(N))
    u += gamma*(G.T@error)
factor = np.max(np.abs(1-gamma*sigma**2))
if relative_step < 1:
    assert np.all(np.diff(energies) <= 1e-11)
assert np.all(np.asarray(energies) <= energies[0]*factor**np.arange(trials+1)+1e-11)
print(f"RMS error from {energies[0]:.5f} to {energies[-1]:.5f}; worst-case error-norm factor {factor:.4f}.")

## Compare a trajectory to a convergence diagnostic
The dashed bound permits mixtures of error directions and can lie above the actual error. Print the singular values to estimate which components of a tracking error are learned slowly.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9.2,3.5),layout="constrained")
ax[0].plot(t,target,color="0.2",lw=2,label="Target")
for j,style in [(0,":"),(min(2,trials),"--"),(trials,"-.")]:
    ax[0].plot(t,paths[j],style,label=f"Trial {j}")
idx=np.arange(trials+1)
ax[1].semilogy(idx,energies,label="Simulated RMS error")
ax[1].semilogy(idx,energies[0]*factor**idx,"--",label="Singular-value bound")
ax[0].set(xlabel="Fraction of one move",ylabel="Normalised position",title="Motion within a trial")
ax[1].set(xlabel="Completed learning updates",ylabel="RMS tracking error",title="Improvement across trials")
for p in ax:
    p.grid(alpha=.2); p.legend(fontsize=8)
plt.show()

## Change one assumption
Set the relative step to 1.05 and increase the trial count. A particular reference can show several improving trials even though the worst singular direction grows. Replace the repeatable disturbance by a different vector on each trial, remove the deterministic contraction assertion, and calculate errors against a separate clean trajectory. Explain why the original bound no longer follows.

## Identify the worst-case error from an independent eigenproblem
The eigenvalues of the symmetric error-propagation matrix are the gains on independent error directions. OneSimulation can solve that eigenproblem locally. In the browser use the corresponding NumPy solver and cross-check against the earlier singular-value prediction.

In [ ]:
error_map = np.eye(N) - gamma * (G @ G.T)
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigenvalues
    method = "OneSimulation symmetric eigenvalues"
except ImportError:
    symmetric_eigenvalues = np.linalg.eigvalsh
    method = "NumPy symmetric eigenvalues"

error_gains = symmetric_eigenvalues(error_map)
measured_factor = np.max(np.abs(error_gains))
assert np.isclose(measured_factor, factor)
print(f"{method}: worst one-trial error gain {measured_factor:.6f}")